# 10.6 · 目标检测 / Object Detection

> **课程定位 / Where this fits**
> 第 6 课，**Part 10 · 计算机视觉**。
> Lesson 6, **Part 10 · Computer Vision**.
>
> 前 5 课都在做**图像分类**（"整张图是什么"）。但现实任务常常是：**一张图里有哪些物体？各自在哪？**——这就是**目标检测**：同时**定位(画框 bbox)** 和**分类**每个物体。自动驾驶、人脸识别、安防都靠它。本课讲清 bbox、**IoU**、**NMS**（都从零实现并可视化），以及 **R-CNN 家族 / YOLO / SSD** 的核心思想与取舍。
> The first 5 lessons did **classification** ("what is the whole image"). But real tasks often ask: **which objects are in the image, and where?** — that's **object detection**: simultaneously **localize (draw bounding boxes)** and **classify** each object. Self-driving, face recognition, surveillance rely on it. We'll cover bbox, **IoU**, **NMS** (implemented from scratch and visualized), plus the core ideas/tradeoffs of **R-CNN family / YOLO / SSD**.
>
> 💼 **实战/面试视角**："IoU 怎么算 / NMS 干什么 / 两阶段 vs 一阶段 / anchor 是什么 / mAP" 是检测岗必考。
> 💼 **Practical/interview angle:** "compute IoU / what NMS does / two-stage vs one-stage / anchors / mAP" — detection-role essentials.

> 📐 **符号约定 / Notation**
> - bbox $(x_1,y_1,x_2,y_2)$ —— 框的左上、右下角坐标 / box corners (top-left, bottom-right)
> - IoU —— 交并比, 两框重叠程度(0~1) / Intersection over Union
> - 置信度(confidence) —— 模型对某框含物体的把握 / box objectness/score

> 💡 **面试相关 / Interview-relevant**
> - "IoU 的定义与计算"（出镜率 ★★★★★）
> - "NMS 非极大值抑制的作用与流程"（★★★★★）
> - "两阶段(Faster R-CNN) vs 一阶段(YOLO/SSD) 区别与取舍"（★★★★★）
> - "anchor box 是什么"（★★★★）
> - "mAP 评价指标"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解检测任务与 bbox 表示。
   Understand detection and bounding-box representation.
2. **从零实现 IoU** 并可视化。
   Implement IoU from scratch and visualize.
3. **从零实现 NMS**，理解为何需要它。
   Implement NMS from scratch; understand why it's needed.
4. 掌握 anchor 思想与两阶段/一阶段检测器的取舍。
   Grasp anchors and the two-stage/one-stage tradeoff.
5. 了解 mAP 评价指标。
   Know the mAP metric.

## 目录 / TOC
1. [检测 = 定位 + 分类 ⭐](#1)
2. [IoU：度量框的重叠（从零）⭐](#2)
3. [NMS：去除重复框（从零）⭐](#3)
4. [Anchor 与检测器家族 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 检测 = 定位 + 分类 ⭐ / Detection = Localization + Classification

**图像分类**输出一个标签（"这是猫"）。**目标检测**要对图里**每个**物体输出：
**Classification** outputs one label ("a cat"). **Detection** outputs, for **each** object:
- 一个**边界框(bounding box, bbox)**：物体的位置，常用 $(x_1,y_1,x_2,y_2)$（左上、右下角）或 $(x,y,w,h)$（中心+宽高）表示。
  A **bounding box (bbox):** the location, often $(x_1,y_1,x_2,y_2)$ (corners) or $(x,y,w,h)$ (center+size).
- 一个**类别**和一个**置信度分数**。
  A **class** and a **confidence score**.

下面在一张图上画几个示意框，建立直觉。
Let's draw a few illustrative boxes to build intuition.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import matplotlib.patches as patches
from skimage import data
sns.set_theme(style="white")

img = data.astronaut()                                  # 一张 512×512 彩色图 / a color image
# 手工标几个示意框 (x1,y1,x2,y2, 类别) / a few illustrative boxes
boxes = [(120, 50, 360, 360, "person"), (250, 5, 400, 120, "helmet"), (300, 300, 460, 500, "suit")]
fig, ax = plt.subplots(figsize=(5.5, 5.5)); ax.imshow(img)
for x1, y1, x2, y2, label in boxes:
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", lw=2)  # 画框 / draw bbox
    ax.add_patch(rect)
    ax.text(x1, y1-5, label, color="black", fontsize=9, bbox=dict(facecolor="lime", alpha=0.8, pad=1))
ax.set_title("目标检测: 为每个物体输出 [框 + 类别 + 置信度]"); ax.axis("off")
plt.tight_layout(); plt.show()
print("bbox 表示: (x1,y1,x2,y2)角点 或 (x,y,w,h)中心+宽高; 检测=定位(框)+分类(类别)")
print("评价: 用 IoU 判断框是否准, 用 mAP 综合衡量(见 §4)")


<a id="2"></a>
## 2. IoU：度量框的重叠（从零）⭐ / IoU: Measuring Box Overlap

怎么判断一个预测框"准不准"？看它和真实框**重叠多少**。**IoU(Intersection over Union, 交并比)** = **交集面积 / 并集面积**，取值 0（完全不重叠）到 1（完全重合）。
How to judge if a predicted box is "good"? Measure its **overlap** with the ground-truth box. **IoU (Intersection over Union)** = **intersection area / union area**, from 0 (no overlap) to 1 (identical).

IoU 是检测里**无处不在**的工具：判定预测对错（如 IoU>0.5 算命中）、NMS 去重、anchor 匹配都用它。下面从零实现并可视化。
IoU is **everywhere** in detection: deciding correctness (e.g. IoU>0.5 = a hit), NMS deduplication, anchor matching. We implement and visualize it.


In [ ]:
def iou(boxA, boxB):
    """计算两个框 (x1,y1,x2,y2) 的 IoU / IoU of two boxes."""
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])      # 交集左上角 / intersection top-left
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])      # 交集右下角 / intersection bottom-right
    inter = max(0, xB - xA) * max(0, yB - yA)                   # 交集面积(无重叠则为0) / intersection area
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter                              # 并集 = 两面积之和 - 交集 / union
    return inter / union if union > 0 else 0

# 可视化三种重叠程度 / visualize three overlap levels
gt = (100, 100, 300, 300)                                       # 真实框 / ground-truth box
preds = [(110, 110, 310, 310), (180, 180, 380, 380), (260, 260, 460, 460)]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, pred in zip(axes, preds):
    ax.add_patch(patches.Rectangle((gt[0],gt[1]), gt[2]-gt[0], gt[3]-gt[1], fill=False, edgecolor="lime", lw=2.5, label="真实框"))
    ax.add_patch(patches.Rectangle((pred[0],pred[1]), pred[2]-pred[0], pred[3]-pred[1], fill=False, edgecolor="red", lw=2.5, label="预测框"))
    ax.set_xlim(50,500); ax.set_ylim(500,50); ax.set_title(f"IoU = {iou(gt, pred):.2f}"); ax.legend(loc="upper right", fontsize=8)
fig.suptitle("IoU = 交集/并集: 越接近1两框越重合; 检测中常以 IoU>0.5 判为命中"); plt.tight_layout(); plt.show()
print("IoU=交集面积/并集面积, 0(不重叠)~1(完全重合)")
print("用途: 判断预测命中(IoU>阈值)/ NMS去重 / anchor与真值匹配; 是检测的基础度量")


<a id="3"></a>
## 3. NMS：去除重复框（从零）⭐ / NMS: Removing Duplicate Boxes

检测器对**同一个物体**往往会吐出**一堆高度重叠的框**（稍微挪一点、缩放一点都被检出）。我们只想保留**最好的那一个**。**NMS(Non-Maximum Suppression, 非极大值抑制)** 就是干这个的，流程：
A detector usually emits **many overlapping boxes for the same object** (slightly shifted/scaled). We want to keep only **the best one**. **NMS (Non-Maximum Suppression)** does this:
1. 按置信度从高到低排序所有框。
   Sort all boxes by confidence, high to low.
2. 取分数最高的框，保留它；把和它 **IoU 超过阈值**的其它框**全部抑制(删掉)**——它们被认为是同一物体的重复检测。
   Take the top box, keep it; **suppress (remove) all others with IoU above a threshold** — they're duplicates of the same object.
3. 在剩下的框里重复，直到没有框可处理。
   Repeat on the rest until none remain.

下面从零实现 NMS，并**可视化"去重前 vs 去重后"**。
We implement NMS from scratch and visualize **before vs after**.


In [ ]:
def nms(boxes, scores, iou_thresh=0.5):
    """非极大值抑制: 返回保留框的索引 / NMS, returns kept indices."""
    idxs = list(np.argsort(scores)[::-1])                 # 按分数从高到低排序 / sort by score desc
    keep = []
    while idxs:
        cur = idxs.pop(0)                                 # 取当前最高分的框 / highest-score box
        keep.append(cur)
        # 抑制掉与它重叠过大的框 / drop boxes overlapping too much with it
        idxs = [i for i in idxs if iou(boxes[cur], boxes[i]) < iou_thresh]
    return keep

# 造一堆重叠的检测框(模拟检测器对2个物体吐出的重复框) / synthetic overlapping detections for 2 objects
np.random.seed(0)
obj1 = [(100+dx, 100+dy, 200+dx, 200+dy) for dx,dy in np.random.randint(-8,8,(6,2))]
obj2 = [(300+dx, 250+dy, 420+dx, 370+dy) for dx,dy in np.random.randint(-8,8,(5,2))]
all_boxes = obj1 + obj2
scores = list(np.random.uniform(0.5, 0.99, len(all_boxes)))
kept = nms(all_boxes, scores, iou_thresh=0.5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, title, show in [(axes[0], f"NMS 之前: {len(all_boxes)} 个重复框", range(len(all_boxes))),
                        (axes[1], f"NMS 之后: 只剩 {len(kept)} 个", kept)]:
    for i in show:
        b = all_boxes[i]
        c = "red" if i in kept else "gray"
        ax.add_patch(patches.Rectangle((b[0],b[1]), b[2]-b[0], b[3]-b[1], fill=False, edgecolor=c, lw=2))
        ax.text(b[0], b[1]-3, f"{scores[i]:.2f}", fontsize=7, color=c)
    ax.set_xlim(50,480); ax.set_ylim(420,50); ax.set_title(title)
plt.tight_layout(); plt.show()
print(f"输入 {len(all_boxes)} 个重叠框 → NMS 后保留 {len(kept)} 个(每个物体只留最高分的那个)")
print("NMS 流程: 按分数排序 → 取最高分 → 抑制与它IoU>阈值的框 → 重复; 几乎所有检测器后处理都用")


<a id="4"></a>
## 4. Anchor 与检测器家族 + 小结 ⭐ / Anchors & Detector Families

**怎么找出候选框？** 现代检测器常用 **anchor(锚框)**：在图像上铺满**预设的不同大小/长宽比的参考框**，网络只需预测"每个 anchor 里有没有物体、该怎么微调它的位置"。这把"在无限多可能的框里搜索"变成"对固定 anchor 做分类+回归"。
**How to propose boxes?** Modern detectors use **anchors**: tile the image with **preset reference boxes of various sizes/aspect ratios**; the net just predicts "is there an object in each anchor, and how to nudge its position." This turns "searching infinitely many boxes" into "classify + regress fixed anchors."


In [ ]:
# 可视化 anchor: 在网格点上铺不同尺度/长宽比的参考框 / visualize anchors at grid points
fig, ax = plt.subplots(figsize=(6, 6)); ax.imshow(data.astronaut())
scales = [40, 80]; ratios = [(1,1), (1,2), (2,1)]          # 2种尺度 × 3种长宽比 = 6种 anchor / scales x ratios
for cx in range(128, 512, 160):                            # 网格中心点 / grid centers
    for cy in range(128, 512, 160):
        for s in scales:
            for rw, rh in ratios:
                w, h = s*rw, s*rh
                ax.add_patch(patches.Rectangle((cx-w/2, cy-h/2), w, h, fill=False, edgecolor="cyan", lw=0.7, alpha=0.7))
ax.plot([cx for cx in range(128,512,160) for _ in range(3)],
        [cy for _ in range(3) for cy in range(128,512,160)], "r+", ms=8)
ax.set_title("Anchor 锚框: 每个网格点铺多种尺度×长宽比的参考框\n网络预测每个anchor: 有无物体 + 位置微调"); ax.axis("off")
plt.tight_layout(); plt.show()
print("anchor: 预设的参考框(多尺度多长宽比), 把'搜索框'变成'对固定anchor分类+回归位置'")


**两大检测器家族**（面试核心对比）：
**Two detector families** (core interview comparison):

| | 两阶段 Two-stage (R-CNN 家族) | 一阶段 One-stage (YOLO / SSD) |
|---|---|---|
| 流程 / Pipeline | ①先生成候选区域(region proposal) ②再对每个区域分类+精修框 | 一步到位: 直接在整图密集预测框+类别 |
| 代表 / Examples | R-CNN → Fast R-CNN → Faster R-CNN | YOLO 系列, SSD, RetinaNet |
| 精度 / Accuracy | 通常更高 (higher) | 略低但差距在缩小 (slightly lower) |
| 速度 / Speed | 较慢 (slower) | **快, 可实时** (fast, real-time) |
| 适用 / Use | 高精度离线场景 | 实时场景(自动驾驶/视频) |

- **R-CNN → Fast → Faster** 的进化主线：把"候选区域生成"从慢的传统方法(选择性搜索)逐步换成**网络自己生成(RPN)**，越来越快。
  **R-CNN → Fast → Faster** evolution: replace slow proposal generation (selective search) with a **network (RPN)**, getting faster.
- **YOLO("You Only Look Once")**：把检测当成**单个回归问题**，整张图一次前向就输出所有框，因此**极快、能实时**。
  **YOLO ("You Only Look Once"):** treats detection as **one regression**, outputting all boxes in a single forward pass — **very fast, real-time**.

**评价指标 mAP**：对每个类别按不同置信度阈值算 precision-recall 曲线下面积(AP)，再对所有类别平均得 **mAP(mean Average Precision)**。常写成 mAP@0.5（IoU 阈值 0.5）。这是检测的标准指标（呼应 Part 7 的 PR 曲线）。
**Metric mAP:** per class, area under the precision-recall curve (AP) across score thresholds, averaged over classes → **mAP (mean Average Precision)**, often mAP@0.5 (IoU threshold 0.5). The standard detection metric (echoing Part 7's PR curves).

> 💡 实战：直接用成熟库——torchvision 的 `fasterrcnn_resnet50_fpn`、Ultralytics 的 YOLO——几行就能加载预训练检测器并微调；很少从零造。
> 💡 Practical: use mature libs — torchvision's `fasterrcnn_resnet50_fpn`, Ultralytics YOLO — load a pretrained detector and fine-tune in a few lines; rarely built from scratch.

```
检测=定位(bbox)+分类+置信度; bbox=(x1,y1,x2,y2)或(x,y,w,h)
IoU=交集/并集(0~1): 判命中/NMS/anchor匹配的基础度量
NMS: 按分数排序→取最高→抑制与它IoU>阈值的框→重复; 去除同物体的重复框
anchor: 预设多尺度多长宽比参考框 → 把搜索变成对anchor分类+回归
两阶段(Faster R-CNN): 先proposal再精修, 精度高较慢; 一阶段(YOLO/SSD): 一次预测, 快可实时
mAP: 检测标准指标(各类AP的平均, 常 mAP@0.5)
```

### 💡 面试速查 / Interview cheat-sheet
1. **IoU**: 交集/并集, 度量两框重叠; >0.5 常判命中。
   IoU: intersection/union; >0.5 often counts as a hit.
2. **NMS**: 排序→取最高→抑制高IoU重复→重复; 去重复框。
   NMS: sort→take top→suppress high-IoU duplicates→repeat.
3. **两阶段 vs 一阶段**: Faster R-CNN(精度高慢) vs YOLO/SSD(快实时)。
   Two-stage vs one-stage: Faster R-CNN (accurate/slow) vs YOLO/SSD (fast/real-time).
4. **anchor**: 预设参考框, 网络做分类+位置回归。
   Anchors: preset reference boxes; net classifies + regresses position.
5. **mAP**: 各类 AP 的平均, 检测标准指标。
   mAP: mean of per-class AP, the standard detection metric.

### 下一节 / Next
**10.7 图像分割**——检测给的是"框"，分割要精确到**每个像素属于哪个物体/类别**。我们会讲语义分割 vs 实例分割、FCN、亲手搭一个 **U-Net**、并理解 Dice loss。
**10.7 Segmentation** — detection gives boxes; segmentation labels **every pixel**. We'll cover semantic vs instance segmentation, FCN, build a **U-Net**, and understand Dice loss.
